In [7]:
from neo4j import GraphDatabase
from google import genai
import os
from dotenv import load_dotenv

In [8]:
main_driver = GraphDatabase.driver(uri=os.getenv("NEO4J_URI"),auth=(os.getenv("NEO4J_USER"),os.getenv("NEO4J_PASS")))

client = genai.Client(api_key=os.getenv("GENAI_API_KEY"))
MODEL = "gemini-2.5-flash"

In [9]:
with open("../prompts/graph_retrieval.txt", "r") as file:
    content = file.read()


def generate_cypher(query):
    prompt = content + query
    
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )
    
    return response.text.strip()

In [10]:
def run_cypher(cypher):
    with main_driver.session() as session:
        result = session.run(cypher)
        return [record.data() for record in result]

In [11]:
def test_query(user_query):
    print("\n🔍 User Query:", user_query)

    cypher = generate_cypher(user_query)

    try:
        results = run_cypher(cypher)
        print("\n📊 Results:")
        for r in results:
            print(r)
    except Exception as e:
        print("❌ Error executing query:", e)

In [16]:
test_query("What are the herbs that can be used to cure Fatigue")


🔍 User Query: What are the herbs that can be used to cure Fatigue

📊 Results:
{'h.name': 'Papaya'}
{'h.name': 'Turmeric'}
{'h.name': 'Black Pepper'}
{'h.name': 'Ginger'}
{'h.name': 'Honey'}
{'h.name': 'Honey'}
{'h.name': 'Garlic'}
{'h.name': 'Tulsi'}
{'h.name': 'Beetroot'}
